# Planar Classification — PyTorch MLP (底层从零实现)

用 PyTorch **最底层 API** 实现单层隐藏层 MLP，**不使用** `nn.Module` / `nn.Parameter` / `nn.BCELoss` / `optim`。

- 用 `torch.tensor(..., requires_grad=True)` 定义 `W1, b1, W2, b2`
- 手写前向传播：`tanh` 隐藏层 + `sigmoid` 输出
- 手写交叉熵损失
- `loss.backward()` 自动求梯度，**手写 SGD** 更新参数

> 云端 GPU：先运行第一个 code cell 安装依赖，再 **Run All**。本 notebook 自包含。

In [ ]:
# 云端环境首次运行：安装依赖（本地已装好可跳过）
# !pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

torch.manual_seed(1)
np.random.seed(1)

## 1. Planar 数据集

In [ ]:
def load_planar_dataset():
    """生成 flower 形 2 类数据集。返回 X:(2,m), Y:(1,m)。"""
    np.random.seed(1)
    m, D, a = 400, 2, 4
    N = m // 2
    X = np.zeros((m, D))
    Y = np.zeros((m, 1), dtype=np.uint8)
    for j in range(2):
        ix = range(N * j, N * (j + 1))
        t = np.linspace(j * 3.12, (j + 1) * 3.12, N) + np.random.randn(N) * 0.2
        r = a * np.sin(4 * t) + np.random.randn(N) * 0.2
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        Y[ix] = j
    return X.T, Y.T


X_np, Y_np = load_planar_dataset()
print("X shape:", X_np.shape, "| Y shape:", Y_np.shape, "| m =", X_np.shape[1])

plt.scatter(X_np[0], X_np[1], c=Y_np[0], s=40, cmap=plt.cm.Spectral)
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Planar dataset")
plt.show()

## 2. 从零定义参数与前向传播

结构：`input(2) → hidden(n_h, tanh) → output(1, sigmoid)`

对应 NumPy 版作业：
- $Z^{[1]} = W^{[1]} X + b^{[1]}$, $A^{[1]} = \tanh(Z^{[1]})$
- $Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$, $\hat{y} = \sigma(Z^{[2]})$
- $J = -\frac{1}{m}\sum_i \bigl[y^{(i)}\log\hat{y}^{(i)} + (1-y^{(i)})\log(1-\hat{y}^{(i)})\bigr]$

In [ ]:
def init_parameters(n_x, n_h, n_y, device, seed=2):
    """用 requires_grad=True 的 Tensor 初始化参数（W*0.01, b=0）。"""
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    params = {
        "W1": torch.randn(n_h, n_x, generator=g, device=device, requires_grad=True) * 0.01,
        "b1": torch.zeros(n_h, 1, device=device, requires_grad=True),
        "W2": torch.randn(n_y, n_h, generator=g, device=device, requires_grad=True) * 0.01,
        "b2": torch.zeros(n_y, 1, device=device, requires_grad=True),
    }
    return params


def forward_propagation(X, params):
    """
    X: (n_x, m)
    返回 A2: (1, m)
    """
    W1, b1, W2, b2 = params["W1"], params["b1"], params["W2"], params["b2"]
    Z1 = W1 @ X + b1
    A1 = torch.tanh(Z1)
    Z2 = W2 @ A1 + b2
    A2 = torch.sigmoid(Z2)
    return A2


def compute_cost(A2, Y):
    """手写二元交叉熵，与 NumPy 版公式一致。"""
    m = Y.shape[1]
    eps = 1e-8
    A2 = torch.clamp(A2, eps, 1 - eps)
    loss = -torch.sum(Y * torch.log(A2) + (1 - Y) * torch.log(1 - A2)) / m
    return loss


def to_tensors(X_np, Y_np, device):
    X = torch.tensor(X_np, dtype=torch.float32, device=device)
    Y = torch.tensor(Y_np, dtype=torch.float32, device=device)
    return X, Y

## 3. 训练（前向 + 手写 Loss + autograd 反向 + 手写 SGD）

In [ ]:
def update_parameters(params, lr=1.2):
    """手写梯度下降：param = param - lr * grad"""
    with torch.no_grad():
        for key in ["W1", "b1", "W2", "b2"]:
            p = params[key]
            p -= lr * p.grad


def zero_grad(params):
    for p in params.values():
        if p.grad is not None:
            p.grad.zero_()


def train_mlp(X_np, Y_np, n_h=4, lr=1.2, num_iterations=10000, print_cost=True, seed=3):
    X, Y = to_tensors(X_np, Y_np, device)
    n_x, n_y = X.shape[0], Y.shape[0]
    params = init_parameters(n_x, n_h, n_y, device, seed=seed)

    costs = []
    for i in range(num_iterations):
        zero_grad(params)

        A2 = forward_propagation(X, params)   # 前向
        loss = compute_cost(A2, Y)            # 损失
        loss.backward()                       # autograd 反向传播

        update_parameters(params, lr=lr)      # 手写 SGD

        costs.append(loss.item())
        if print_cost and i % 1000 == 0:
            print(f"Cost after iteration {i:5d}: {loss.item():.6f}")

    return params, costs


params, costs = train_mlp(X_np, Y_np, n_h=4, lr=1.2, num_iterations=10000)

plt.plot(costs)
plt.xlabel("iteration"); plt.ylabel("BCE loss"); plt.title("Training loss")
plt.show()

## 4. 预测与准确率

In [ ]:
@torch.no_grad()
def predict(params, X_np):
    X = torch.tensor(X_np, dtype=torch.float32, device=device)
    probs = forward_propagation(X, params)
    return (probs > 0.5).float().cpu().numpy()


predictions = predict(params, X_np)
accuracy = (np.dot(Y_np, predictions) + np.dot(1 - Y_np, 1 - predictions)) / Y_np.size * 100
print(f"Accuracy: {accuracy:.0f}%")

## 5. 决策边界可视化

In [ ]:
def plot_decision_boundary(params, X, y):
    x_min, x_max = X[0].min() - 1, X[0].max() + 1
    y_min, y_max = X[1].min() - 1, X[1].max() + 1
    h = 0.01
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()].T  # (2, N)

    with torch.no_grad():
        Xg = torch.tensor(grid, dtype=torch.float32, device=device)
        Z = (forward_propagation(Xg, params) > 0.5).float().cpu().numpy().reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    plt.scatter(X[0], X[1], c=y, s=40, cmap=plt.cm.Spectral, edgecolors="k")
    plt.xlabel("x1"); plt.ylabel("x2")


plot_decision_boundary(params, X_np, Y_np[0])
plt.title("PyTorch MLP (low-level) — hidden size 4")
plt.show()

## 6.（可选）不同隐藏层大小对比

In [ ]:
hidden_sizes = [1, 2, 3, 4, 5, 20]
plt.figure(figsize=(14, 8))

for i, n_h in enumerate(hidden_sizes):
    p, _ = train_mlp(X_np, Y_np, n_h=n_h, num_iterations=5000, print_cost=False, seed=3 + i)
    pred = predict(p, X_np)
    acc = (np.dot(Y_np, pred) + np.dot(1 - Y_np, 1 - pred)) / Y_np.size * 100

    plt.subplot(2, 3, i + 1)
    plot_decision_boundary(p, X_np, Y_np[0])
    plt.title(f"n_h={n_h}, acc={acc:.0f}%")

plt.tight_layout()
plt.show()